In [4]:
import math
import numpy as np
from scipy.optimize import minimize

import qnexus as qnx

from guppylang import guppy
from guppylang.std.builtins import array, result
from guppylang.std.quantum import qubit, h, cx, ry, rz, measure_array
from guppylang.std.angles import angle
from guppylang.std.builtins import comptime

from selene_sim import build, Quest

# ------------------------------------------------------------------
# Problem setup: TFIM Hamiltonian  H = -J sum ZiZ_{i+1} - h sum Xi
# ------------------------------------------------------------------
J = 1.0
N_SHOTS = 2000
#N_QUBITS = 6
#N_LAYERS = 2
#J = 1.0
#H_FIELD = 0.5 * J
#N_SHOTS = 2000
#N_PARAMS = 2 * N_QUBITS * (N_LAYERS + 1)   # Ry+Rz per qubit per layer, plus final layer


# ------------------------------------------------------------------
# 0. Nexus auth + project (one-time)
# ------------------------------------------------------------------

# ------------------------------------------------------------------
# 1. Hardware-Efficient Ansatz, built fresh for a given parameter
#    vector. Guppy entry points can't take runtime arguments (must be
#    zero-argument), so parameters are baked in as Python-level
#    closure constants and the whole program is recompiled per call --
#    same pattern as make_circuit() in tfim_guppy.py.
# ------------------------------------------------------------------
def make_hea_circuit(params, N, N_LAYERS, measure_x: bool):

    @guppy.comptime
    def circuit() -> None:
        qs = array(qubit() for _ in range(comptime(N)))

        for layer in range(N_LAYERS):
            for i in range(N):
                ry(qs[i], angle(params[layer * 2 * N + 2 * i] / math.pi))
                rz(qs[i], angle(params[layer * 2 * N + 2 * i + 1] / math.pi))
            for i in range(N - 1):
                cx(qs[i], qs[i + 1])
            cx(qs[N - 1], qs[0])  # cierra el anillo -- necesario para el Hamiltoniano periodico

        # final rotation layer (no entangler after it)
        final_offset = N_LAYERS * 2 * N
        for i in range(N):
            ry(qs[i], angle(params[final_offset + 2 * i] / math.pi))
            rz(qs[i], angle(params[final_offset + 2 * i + 1] / math.pi))

        if measure_x:
            for i in range(N):
                h(qs[i])

        bits = measure_array(qs) #Instruction to measure all qubits 
        result("bits", bits)

    return circuit


# ------------------------------------------------------------------
# 2. Energy evaluation via Nexus: submit BOTH the Z-basis circuit
#    (gives all ZiZ_{i+1} terms) and the X-basis circuit (gives all
#    Xi terms) as ONE batched job to minimize queue round-trips.
# ------------------------------------------------------------------
def evaluate_energy_nexus(params, tag=""):
    qnx.login()  

    project = qnx.projects.get_or_create("TFIM-VQE-HEA")
    qnx.context.set_active_project(project)

    emulator_config = qnx.models.HeliosConfig(
        system_name="Helios-1E-lite",
        emulator_config=qnx.models.HeliosEmulatorConfig(n_qubits=N_QUBITS),
    )

    circuit_z = make_hea_circuit(params, N,  measure_x=False)
    circuit_x = make_hea_circuit(params, N, measure_x=True)

    hugr_z = qnx.hugr.upload(hugr_package=circuit_z.compile(), name=f"HEA_Z_{tag}")
    hugr_x = qnx.hugr.upload(hugr_package=circuit_x.compile(), name=f"HEA_X_{tag}")

    job_ref = qnx.start_execute_job(
        programs=[hugr_z, hugr_x],
        n_shots=[N_SHOTS, N_SHOTS],
        backend_config=emulator_config,
        name=f"VQE eval {tag}",
    )
    qnx.jobs.wait_for(job_ref)
    result_refs = qnx.jobs.results(job_ref)

    # .download_result() gives a QsysResult; iterating it yields QsysShot
    # objects, each with .as_dict() -> {"bits": [0,1,...]} for this
    # circuit (we tagged the measurement with result("bits", ...)).
    # This is the raw shot table -- no counts/histogram, one row per shot.
    shots_z = [shot.as_dict()["bits"] for shot in result_refs[0].download_result()]

    shots_x = [shot.as_dict()["bits"] for shot in result_refs[1].download_result()]

    bits_z = np.array(shots_z)
    bits_x = np.array(shots_x)

    spins_z = 1 - 2 * bits_z   # (shots, N_QUBITS), values in {+1,-1}
    spins_x = 1 - 2 * bits_x

    zz_nn = np.mean([
        (spins_z[:, i] * spins_z[:, (i + 1) % N]).mean()
        for i in range(N)
    ])
    mean_z = spins_z.mean()
    mean_x = spins_x.mean()

    energy = -J * N * zz_nn - H_FIELD * N * mean_x
    return energy, mean_z, zz_nn, mean_x


# Optional local fallback (uncomment usage in the optimizer loop below
# if Nexus queueing makes per-iteration submission impractical):
def evaluate_energy_local(params, N, N_LAYERS, H_FIELD):
    from selene_sim import build, Quest
    from hugr.qsystem.result import QsysResult

    def run(circuit): #THIS FUNCTION RUNS THE SIMULATION ON THE LOCAL SELENE SIMULATOR AND RETURNS THE BITS
        runner = build(circuit.compile())
        sim_result = QsysResult(runner.run_shots(simulator=Quest(), n_qubits=N, n_shots=2000))
        return np.array([s.as_dict()["bits"] for s in sim_result])

    bits_z = run(make_hea_circuit(params, N, N_LAYERS, measure_x=False))
    bits_x = run(make_hea_circuit(params, N, N_LAYERS, measure_x=True))
    

    spins_z = 1 - 2 * bits_z
    spins_x = 1 - 2 * bits_x
    zz_nn = np.mean([(spins_z[:, i] * spins_z[:, (i + 1) % N]).mean() for i in range(N)])
    mean_z = spins_z.mean()
    mean_x = spins_x.mean()
    energy = -J * N * zz_nn - H_FIELD * N * mean_x
    return energy, mean_z, zz_nn, mean_x

def mean_z_overall(counts):
    """Average <Z> across all qubits and all shots."""
    total_shots = sum(counts.values())
    n_qubits = len(next(iter(counts))[0][1])
    acc = 0
    for key, n in counts.items():
        bitstring = key[0][1]
        for bit in bitstring:
            z = 1 if bit == '0' else -1
            acc += z * n
    return acc / (total_shots * n_qubits)

def mean_zi_zi1_correlator(counts):
    """Promedio de <Zi*Zi+1> sobre todos los pares de qubits adyacentes y todos los shots."""
    total_shots = sum(counts.values())
    n_qubits = len(next(iter(counts))[0][1])
    n_pairs = n_qubits - 1
    acc = 0
    for key, n in counts.items():
        bitstring = key[0][1]
        for i in range(n_pairs):
            zi = 1 if bitstring[i] == '0' else -1
            zi1 = 1 if bitstring[i + 1] == '0' else -1
            acc += (zi * zi1) * n
    return acc / (total_shots * n_pairs)



def evaluate_energy_NEXUS_selene(params):
    project = qnx.projects.get_or_create("VQE-Selene")
    qnx.context.set_active_project(project)
    
    circuit_z = make_hea_circuit(params, False)
    circuit_x = make_hea_circuit(params, True)

    hugr_z = qnx.hugr.upload(
            hugr_package=circuit_z.compile(),
            name="VQE-z",
        )
    hugr_x = qnx.hugr.upload(
            hugr_package=circuit_x.compile(),
            name="VQE-x",
        )
    
    config = qnx.models.SeleneConfig(
            n_qubits=6, 
            simulator=qnx.models.StatevectorSimulator()
        )
    
    selene_quest_job_ref = qnx.start_execute_job(
            programs=[hugr_z, hugr_x],
            n_shots=[2000, 2000],
            backend_config=config,
            name=f"VQE_HEA {config.__class__.__name__} {datetime.now()}",
        )
    
        # Might take time during busy periods
    qnx.jobs.wait_for(selene_quest_job_ref, timeout=None)
        
    qsys_result_z = qnx.jobs.results(selene_quest_job_ref)[0].download_result()
    qsys_result_x = qnx.jobs.results(selene_quest_job_ref)[1].donwload_result()
    
    mean_z = mean_z_overall(qsys_result_z.collated_counts())
    mean_x = mean_z_overall(qsys_result_x.collated_counts())
    energy = -J * N_QUBITS * zz_nn - H_FIELD * N_QUBITS * mean_x
    return energy, mean_z, zz_nn, mean_x


# ------------------------------------------------------------------
# 3. Classical optimization loop (gradient-free, since we only get
#    shot-noisy energy estimates back -- COBYLA is a standard VQE
#    choice for this).
# ------------------------------------------------------------------
iteration_log = []

def cost_fn(params, N, N_LAYERS, H_FIELD):
    energy, mean_z, zz_nn, mean_x = evaluate_energy_local(params, N, N_LAYERS, H_FIELD) # tag=str(len(iteration_log)
    iteration_log.append((energy, mean_z, zz_nn, mean_x))
    print(f"iter {len(iteration_log):3d}  E={energy:+.4f}  <Z>={mean_z:+.4f}  "
          f"<ZiZj>_nn={zz_nn:+.4f}  <X>={mean_x:+.4f}")
    return energy



def cost_fn2(params, N, H_FIELD):
    energy, mean_z, zz_nn, mean_x = evaluate_energy_nexus(params, N, H_FIELD, tag=str(len(iteration_log))) # tag=str(len(iteration_log)
    iteration_log.append((energy, mean_z, zz_nn, mean_x))
    print(f"iter {len(iteration_log):3d}  E={energy:+.4f}  <Z>={mean_z:+.4f}  "
          f"<ZiZj>_nn={zz_nn:+.4f}  <X>={mean_x:+.4f}")
    return energy


In [15]:
def get_results(N, h):
    N_QUBITS = N
    N_LAYERS = 2
    H_FIELD = h
    N_PARAMS = 2 * N_QUBITS * (N_LAYERS + 1)  
    rng = np.random.default_rng(0)
    x0 = rng.uniform(-0.4, 0.4, size=N_PARAMS)   # start near |0...0>

    res = minimize(cost_fn, x0, args=(N, N_LAYERS, H_FIELD), method="COBYLA", options={"maxiter": 100, "rhobeg": 0.3})
    final_energy, final_Z, final_ZZ, final_X = iteration_log[-1]
    np.savez_compressed(f'Final state estimates: N ={N} h = {H_FIELD}', E=final_energy, Z=final_Z, ZZ=final_ZZ, X=final_X)
    
    return final_energy, final_Z, final_ZZ, final_X
    #print(f"\nFinal state estimates:")
    #print(f"\n N  = {N_QUBITS} and h = {H_FIELD}:")   
    #print(f"  <Z>              = {final_Z:+.4f}")
    #print(f"  <Z_i Z_{{i+1}}>_nn = {final_ZZ:+.4f}")
    #print(f"  <X>              = {final_X:+.4f}")
    #print(f"  Energy           = {final_energy:+.4f}")


def get_results_Nexus(N, h):
    rng = np.random.default_rng(0)
    x0 = rng.uniform(-0.1, 0.1, size=N_PARAMS)   # start near |0...0>

    res = minimize(cost_fn2, x0, args =(N,H_FIELD), method="COBYLA", options={"maxiter": 50, "rhobeg": 0.3})
    final_energy, final_Z, final_ZZ, final_X = iteration_log[-1]
    np.savez_compressed(f'Final state estimates helios: N ={N} h = {h}', E=final_energy, Z=final_Z, ZZ=final_ZZ, X=final_X)
    print(f"\nFinal state estimates:")
    print(f"\n N  = {N_QUBITS} and h = {H_FIELD}:")   
    print(f"  <Z>              = {final_Z:+.4f}")
    print(f"  <Z_i Z_{{i+1}}>_nn = {final_ZZ:+.4f}")
    print(f"  <X>              = {final_X:+.4f}")
    print(f"  Energy           = {final_energy:+.4f}")

In [12]:
get_results(6, 0.375)

iter 155  E=-4.9413  <Z>=+0.7833  <ZiZj>_nn=+0.7917  <X>=+0.0850
iter 156  E=-4.8888  <Z>=+0.7672  <ZiZj>_nn=+0.7787  <X>=+0.0963
iter 157  E=-5.0275  <Z>=+0.7947  <ZiZj>_nn=+0.8077  <X>=+0.0807
iter 158  E=-5.3077  <Z>=+0.8145  <ZiZj>_nn=+0.8470  <X>=+0.1003
iter 159  E=-5.3328  <Z>=+0.8123  <ZiZj>_nn=+0.8557  <X>=+0.0883
iter 160  E=-4.6130  <Z>=+0.7533  <ZiZj>_nn=+0.7393  <X>=+0.0787
iter 161  E=-5.2381  <Z>=+0.7942  <ZiZj>_nn=+0.8393  <X>=+0.0898
iter 162  E=-5.0639  <Z>=+0.7783  <ZiZj>_nn=+0.8087  <X>=+0.0942
iter 163  E=-5.3120  <Z>=+0.8042  <ZiZj>_nn=+0.8583  <X>=+0.0720
iter 164  E=-5.1541  <Z>=+0.7828  <ZiZj>_nn=+0.8223  <X>=+0.0978
iter 165  E=-5.2871  <Z>=+0.8008  <ZiZj>_nn=+0.8470  <X>=+0.0912
iter 166  E=-5.0427  <Z>=+0.7035  <ZiZj>_nn=+0.8063  <X>=+0.0910
iter 167  E=-5.3582  <Z>=+0.8212  <ZiZj>_nn=+0.8617  <X>=+0.0837
iter 168  E=-5.1567  <Z>=+0.7253  <ZiZj>_nn=+0.8203  <X>=+0.1043
iter 169  E=-5.3246  <Z>=+0.8065  <ZiZj>_nn=+0.8500  <X>=+0.0998
iter 170  E=-5.3690  <Z>=

(np.float64(-6.00075),
 np.float64(0.7896666666666666),
 np.float64(0.945),
 np.float64(0.147))

In [17]:
exact_tfim_observables(6, 1, 0.5)

(np.float64(-6.384694563603674),
 np.float64(-5.962267716578632e-13),
 np.float64(0.931516776847425),
 np.float64(0.26519796750637686))

## Barrido de H_FIELD y comparacion con diagonalizacion exacta

Recorre `h` de 0 a 2 llamando a `get_results(N, h)` (VQE) y comparando contra la diagonalizacion exacta del Hamiltoniano TFIM periodico, para visualizar la transicion de fase alrededor de `h = J`.

In [16]:
import numpy as np
import matplotlib.pyplot as plt

def exact_tfim_observables(N, J, h, periodic=True):
    """Diagonalizacion exacta del TFIM; retorna E0 y observables del estado base."""
    sx = np.array([[0, 1], [1, 0]])
    sz = np.array([[1, 0], [0, -1]])
    I = np.eye(2)

    def op_on(op, site, N):
        mats = [I] * N
        mats[site] = op
        out = mats[0]
        for m in mats[1:]:
            out = np.kron(out, m)
        return out

    dim = 2 ** N
    H = np.zeros((dim, dim))
    n_bonds = N if periodic else N - 1
    for i in range(n_bonds):
        j = (i + 1) % N
        H += -J * op_on(sz, i, N) @ op_on(sz, j, N)
    for i in range(N):
        H += -h * op_on(sx, i, N)

    evals, evecs = np.linalg.eigh(H)
    gs = evecs[:, 0]
    E0 = evals[0]

    mean_z = np.mean([gs @ op_on(sz, i, N) @ gs for i in range(N)])
    zz_nn = np.mean([gs @ (op_on(sz, i, N) @ op_on(sz, (i + 1) % N, N)) @ gs for i in range(n_bonds)])
    mean_x = np.mean([gs @ op_on(sx, i, N) @ gs for i in range(N)])

    return E0, mean_z, zz_nn, mean_x


In [ ]:
def sweep_h_field(N, h_values):
    """Corre VQE + diagonalizacion exacta para un rango de h."""
    vqe = {"h": [], "E": [], "Z": [], "ZZ": [], "X": []}
    exact = {"h": [], "E": [], "Z": [], "ZZ": [], "X": []}

    for h in h_values:
        iteration_log.clear()  # limpiar el log entre corridas de get_results
        E_v, Z_v, ZZ_v, X_v = get_results(N, h)
        for d, val in zip((vqe["E"], vqe["Z"], vqe["ZZ"], vqe["X"]), (E_v, Z_v, ZZ_v, X_v)):
            d.append(val)
        vqe["h"].append(h)

        E_e, Z_e, ZZ_e, X_e = exact_tfim_observables(N, J=1.0, h=h, periodic=True)
        for d, val in zip((exact["E"], exact["Z"], exact["ZZ"], exact["X"]), (E_e, Z_e, ZZ_e, X_e)):
            d.append(val)
        exact["h"].append(h)

        print(f"h={h:.2f}  VQE: E={E_v:+.4f} <Z>={Z_v:+.4f} <ZZ>={ZZ_v:+.4f} <X>={X_v:+.4f}  |  "
              f"Exacto: E={E_e:+.4f} <Z>={Z_e:+.4f} <ZZ>={ZZ_e:+.4f} <X>={X_e:+.4f}")

    return vqe, exact


In [14]:
def plot_sweep(vqe, exact, N):
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    panels = [
        (axes[0, 0], "E", "Energia", "tab:blue"),
        (axes[0, 1], "Z", "<Z>", "tab:orange"),
        (axes[1, 0], "ZZ", "<Zi Zi+1>", "tab:green"),
        (axes[1, 1], "X", "<X>", "tab:red"),
    ]
    for ax, key, label, color in panels:
        ax.plot(exact["h"], exact[key], "k-", label="Exacto")
        ax.plot(vqe["h"], vqe[key], "o--", color=color, label="VQE")
        ax.axvline(1.0, color="gray", linestyle=":", linewidth=1, label="h=J (critico)")
        ax.set_xlabel("h"); ax.set_ylabel(label); ax.legend(); ax.set_title(label)

    plt.suptitle(f"Barrido TFIM periodico, N={N}")
    plt.tight_layout()
    plt.savefig(f"tfim_phase_transition_N{N}.png", dpi=150)
    plt.show()


In [ ]:
h_values = np.linspace(0, 2, 11)   # 0.0, 0.2, ..., 2.0
vqe_results, exact_results = sweep_h_field(6, h_values)
plot_sweep(vqe_results, exact_results, N=6)
